In [ ]:
from option_analyzer import *
from fidelity_positions import FidelityPositions
self = OptionAnalyzer('quotes', 'chain')

In [ ]:
os.system('sync > /dev/null 2>&1')
symlist = self.get_updated_symbol_list(age_ub=60)
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
df_earning = self.count_days_from_earning_reports(df_quotes)
print('Days to E:', self.d2e)

In [ ]:
df_raw = self.build_option_df(symlist)
px.bar(self.check_data_age(df_raw), barmode='group', height=300).show()
dfcp = self.concat_put_call_options(df_raw)
_df = dfcp[(dfcp.OpenInterest > 0) & (dfcp.Bid > 0)].loc[:, ['symbol', 'type', 'ImpVola']].groupby(['symbol', 'type']).mean()
px.bar(_df.reset_index().sort_values(by='ImpVola', ascending=False), x='symbol', y='ImpVola', color='type', barmode='group').show()
_df = dfcp[(dfcp.OpenInterest > 0) & (dfcp.Bid > 0)].loc[:, ['symbol', 'type', 'Volume', 'OpenInterest']].groupby(['symbol', 'type']).sum()
_df['turnover'] = _df['Volume']/_df['OpenInterest']
px.bar(_df.reset_index().sort_values(by='turnover', ascending=False), x='symbol', y='turnover', color='type', barmode='group').show()

### Read Portfolio csv file

In [ ]:
pp_csv = sorted(glob('fidelity/Portfolio*.csv'), key=os.path.getmtime)[-1]
fidpos = FidelityPositions(pp_csv)
df_pos = fidpos.get_option_positions()
df_pos['expDt'] = df_pos['expDt'].astype(str)
drop_cols = ['Bid', 'Ask', 'Volume', 'TimeValue', 'IntrinsicValue', 'cluster', 'dte_cluster', 'Theta', 'Rho'] + ['Last Price', 'Cost Basis Total']
df_pos = df_pos.merge(dfcp, on=['symbol', 'type', 'strike', 'expDt'], how='left').drop(columns=drop_cols)
print('Total value:', fidpos.total_value)
print('Total cash: ', fidpos.total_cash)
print('Premium from selling puts:', fidpos.sum_sell_put_premium(df_pos))
fidpos.option_position_pies(df_pos)
df_pos

In [ ]:
df_dict = self.compute_time_decay_metrics_for_positions(df_pos, dfcp)
df_dict['P'].sort_values(by='hdteProfit', ascending=False)

In [ ]:
df_dict['C'].sort_values(by='hdte_resid')

In [ ]:
df_put = self.select_options_by_type(df_raw, 'put')
pp = ParallelOptionCalculator(df_put, self, f'/run/user/{os.getuid()}/time_decay')
csv_files = pp.do_all_theta_curves(symlist)
dfp = pp.assemble_time_decay_df(csv_files)

### Make sure the two methods yield the same result

In [ ]:
df_call = self.select_options_by_type(df_raw, 'call')
pp = ParallelOptionCalculator(df_call, self, f'/run/user/{os.getuid()}/time_decay')
csv_files = pp.do_all_theta_curves(symlist)
dfc = pp.assemble_time_decay_df(csv_files)

In [ ]:
px.scatter(df_put[(df_put.symbol=='SPY') & (df_put.moneyness >= 0.95) & (df_put.moneyness <= 1) & (df_put.dte <= 60)], x='dte', y='OpenInterest', color='strike', height=1000)

In [ ]:
_f = (df_call.symbol=='QQQ') & (df_call.moneyness >= 0.75) & (df_call.moneyness <= 0.9) & (df_call.dte >= 90) & (df_call.dte <= 270)
px.scatter(df_call[_f], x='dte', y='OpenInterest', color='strike', height=1000)

In [ ]:
_df = self.calc_spread_stats(df_put)
px.bar(_df.sort_values(by='mean spread'), barmode='group', width=60*len(symlist))

In [ ]:
def get_rows(df, symbol, strike, dte):
    _f = (df.symbol==symbol) & (df.strike==strike)
    if dte is not None:
        _f = _f & (df.dte==dte)
    return df[_f]

In [ ]:
dfc_diff = dfc.set_index(['symbol', 'strike', 'dte']).drop(columns=['expDt']) - dfc.set_index(['symbol', 'strike', 'dte']).drop(columns=['expDt'])

In [ ]:
pd.DataFrame({'diff': dfc_diff[np.abs(dfc_diff) > 1e-7].count(), 'same': dfc_diff[dfc_diff==0].count()})

In [ ]:
import random

In [ ]:
dfp_diff[(dfp_diff.dtz != 0)].shape

In [ ]:
def show_diff(symbol, strike, dte):
    return pd.concat([get_rows(dfp, symbol, strike, dte), get_rows(dfp2, symbol, strike, dte)])

In [ ]:
np.abs(dfp_diff[(dfp_diff.dtz != 0)]).sort_values(by='dtz').tail(60)

In [ ]:
show_diff('CRCL', 250, 18)

In [ ]:
worst_cases = np.abs(dfp_diff).idxmax(axis=0)
[(col, idx) for col, idx in worst_cases.to_dict().items()]

In [ ]:
get_rows(df_put, 'CRCL', 250, None)#, 18)

In [ ]:
df_thc = self.get_theta_curve(df_put, 'CRCL', 250)
df_thc = self.setup_trapezoidal_decay(df_thc)

In [ ]:
df_thc

In [ ]:
_df_thc = self.prepare_theta_curve(df_thc[df_thc.dte <= 18].copy(), 355, 0.235)

In [ ]:
_df_thc#.dropna()

In [ ]:
self.shortcut_time_decay(df_thc, 'CRCL', 250, 18, 191.125) #dth, dtz, hdte_resid, resid

In [ ]:
self.compute_time_decay_metrics2(df_thc, 'CRCL', 250, 18, 191.125)

In [ ]:
self.compute_time_decay_metrics(df_put, 'CRCL', 18, 250, debug=True)

### Put options with no earning date on or before expiration date

In [ ]:
hdte_resid_ub = 0.75
spread_ub = 5
moneyness_ub = 0.98
premium_lb = 1
delta_lb = -0.15
_filter = (dfp.moneyness <= moneyness_ub) & (dfp.pctSpread <= spread_ub) & (dfp.hdte_resid<=hdte_resid_ub) & (dfp.E.isna() |(dfp.E > dfp.dte))
_filter = _filter & (dfp.premium >= premium_lb) & (dfp.Delta >= delta_lb)
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False)
print(_dfp.shape)
_dfp.head(60)

In [ ]:
_df = dfcp[(dfcp.type=='P') & (dfcp.symbol=='SPY')]
for s in _df.strike.unique():
    print(s, self.get_theta_curve(dfcp, 'SPY', s, 'P').set_index('dte').to_dict())